# HW09 Part A :: Hyper Parameter Optimization

COSC 6373 -- Adam Nelson-Archer, 2140122

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import optuna

## 1. Load the dataset
a. Use the Iris Dataset from scikit-learn
b. Contains: 150 samples, 3 classes, 4 numerical features
c. Import using load_iris()

In [2]:
iris = load_iris()
X = iris.data
y = iris.target

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Classes: {iris.target_names}")

Features shape: (150, 4)
Labels shape: (150,)
Classes: ['setosa' 'versicolor' 'virginica']


## 2. Split the data
Train (70%), Validation (15%), Test (15%)

In [3]:
# First split into train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Then split the temp into validation (50% of 30% = 15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Training set: {X_train.shape[0]} samples ({(X_train.shape[0]/X.shape[0])*100:.0f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({(X_val.shape[0]/X.shape[0])*100:.0f}%)")
print(f"Test set: {X_test.shape[0]} samples ({(X_test.shape[0]/X.shape[0])*100:.0f}%)")

Training set: 105 samples (70%)
Validation set: 22 samples (15%)
Test set: 23 samples (15%)


## 3. Set up the model & 5. A. Grid Search
KNN Model with GridSearchCV

In [4]:
# define the parameter grid for the grid search
param_grid = {
    'n_neighbors': list(range(1, 21)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# create a k-Nearest Neighbors classifier
knn = KNeighborsClassifier()

# create the grid search object
grid_search = GridSearchCV(knn, param_grid, cv=5)

# Record time
start_time_grid = time.time()

# fit the grid search to the data
grid_search.fit(X_train, y_train)

end_time_grid = time.time()
grid_time = end_time_grid - start_time_grid

# evaluate the model on the validation set
grid_val_accuracy = grid_search.score(X_val, y_val)

# evaluate the model on the test set
grid_test_accuracy = grid_search.score(X_test, y_test)

print("--- Grid Search Results ---")
print("Best parameters: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)
print("Validation set accuracy: ", grid_val_accuracy)
print("Test set accuracy: ", grid_test_accuracy)
print(f"Time taken: {grid_time:.4f} seconds")

--- Grid Search Results ---
Best parameters:  {'metric': 'euclidean', 'n_neighbors': 18, 'weights': 'distance'}
Best cross-validation score:  0.9619047619047618
Validation set accuracy:  1.0
Test set accuracy:  1.0
Time taken: 1.3023 seconds


## 5. B. Random Search
RandomizedSearchCV

In [5]:
# create the random search object
random_search = RandomizedSearchCV(knn, param_grid, n_iter=20, cv=5, random_state=42)

start_time_random = time.time()

# fit the random search to the data
random_search.fit(X_train, y_train)

end_time_random = time.time()
random_time = end_time_random - start_time_random

# evaluate the model on the validation set
random_val_accuracy = random_search.score(X_val, y_val)

# evaluate the model on the test set
random_test_accuracy = random_search.score(X_test, y_test)

print("--- Random Search Results ---")
print("Best parameters: ", random_search.best_params_)
print("Best cross-validation score: ", random_search.best_score_)
print("Validation set accuracy: ", random_val_accuracy)
print("Test set accuracy: ", random_test_accuracy)
print(f"Time taken: {random_time:.4f} seconds")

--- Random Search Results ---
Best parameters:  {'weights': 'uniform', 'n_neighbors': 16, 'metric': 'manhattan'}
Best cross-validation score:  0.9619047619047618
Validation set accuracy:  1.0
Test set accuracy:  1.0
Time taken: 0.4137 seconds


## 5. C. Optuna Optimization

In [6]:
# define an objective function to optimize KNN performance
def objective(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 1, 20)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan'])
    
    knn_opt = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric)
    
    # We use 5-fold cross-validation on the training set to evaluate the hyperparameters
    from sklearn.model_selection import cross_val_score
    score = cross_val_score(knn_opt, X_train, y_train, cv=5).mean()
    return score

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')

start_time_optuna = time.time()

# Run a study (30 trials)
study.optimize(objective, n_trials=30)

end_time_optuna = time.time()
optuna_time = end_time_optuna - start_time_optuna

best_params_optuna = study.best_params
best_score_optuna = study.best_value

# Train the best model on the full training data
best_knn_optuna = KNeighborsClassifier(**best_params_optuna)
best_knn_optuna.fit(X_train, y_train)

optuna_val_accuracy = best_knn_optuna.score(X_val, y_val)
optuna_test_accuracy = best_knn_optuna.score(X_test, y_test)

print("--- Optuna Optimization Results ---")
print("Best parameters: ", best_params_optuna)
print("Best cross-validation score: ", best_score_optuna)
print("Validation set accuracy: ", optuna_val_accuracy)
print("Test set accuracy: ", optuna_test_accuracy)
print(f"Time taken: {optuna_time:.4f} seconds")

--- Optuna Optimization Results ---
Best parameters:  {'n_neighbors': 14, 'weights': 'uniform', 'metric': 'manhattan'}
Best cross-validation score:  0.9619047619047618
Validation set accuracy:  1.0
Test set accuracy:  1.0
Time taken: 0.6363 seconds


## 7. Evaluate Performance Summary

In [7]:
print(f"{'Method':<15} | {'Validation Acc':<15} | {'Test Acc':<10} | {'Time (s)':<10}")
print("-" * 55)
print(f"{'Grid Search':<15} | {grid_val_accuracy:<15.4f} | {grid_test_accuracy:<10.4f} | {grid_time:<10.4f}")
print(f"{'Random Search':<15} | {random_val_accuracy:<15.4f} | {random_test_accuracy:<10.4f} | {random_time:<10.4f}")
print(f"{'Optuna':<15} | {optuna_val_accuracy:<15.4f} | {optuna_test_accuracy:<10.4f} | {optuna_time:<10.4f}")

Method          | Validation Acc  | Test Acc   | Time (s)  
-------------------------------------------------------
Grid Search     | 1.0000          | 1.0000     | 1.3023    
Random Search   | 1.0000          | 1.0000     | 0.4137    
Optuna          | 1.0000          | 1.0000     | 0.6363    


## 8. Reflection

**a. Why do we separate validation and test sets?**
We separate them so that we can use the validation set to tune the hyperparameters of the model without touching the test set. If we tune on the test set, the model's performance on it would be overly optimistic and not representative of true generalization to unseen data. The test set must remain entirely unseen during the training and tuning processes to serve as a final, unbiased evaluation metric.

**b. What happens if you tune hyperparameters using the test set?**
This leads to "data leakage" or overfitting to the test set. The model hyperparameters become specifically optimized for the test set, making the evaluation biased and unrepresentative of how the model will perform on completely new, real-world data.

**c. Compare the three methods:**
- **Which method gave the best accuracy?** 
All three methods likely yielded similar or identical high accuracy because the Iris dataset is very small and relatively simple. However, Grid Search guarantees finding the best combination within the defined grid, while Random and Optuna might find it faster if the search space was larger. 
- **Which method was fastest?**
Random Search is typically the fastest since it evaluates a fixed (smaller) number of iterations. Optuna can also be very fast but has slight overhead for the Bayesian optimization steps. Grid search is the slowest since it performs an exhaustive search (though on this small dataset, the difference is negligible).
- **Which method was easiest to implement?**
Grid Search and Random Search were the easiest because they are built directly into `scikit-learn` and require very similar boilerplate code. Optuna required defining a custom objective function and setting up a study.

**d. Did changing hyperparameters significantly affect accuracy?**
On a simple dataset like Iris, the default parameters often perform very well, so hyperparameter tuning might only provide a small improvement or no improvement if the default is already optimal. However, in general, finding the right `n_neighbors` or distance metric can prevent underfitting or overfitting, leading to better generalization.

**e. Which method would you choose in a real-world scenario and why?**
I would choose **Optuna**. While Grid and Random search are easy to use, they do not scale well to models with many hyperparameters (like Neural Networks or Gradient Boosting trees) or large datasets due to the curse of dimensionality and time costs. Optuna uses Bayesian optimization to focus the search intelligently on promising areas of the hyperparameter space, finding better parameters in fewer iterations.


## Acknowledgment

I used a coding assistant (GPT-5.3-Codex) to help scaffold and organize this notebook.
